# Question 10: Deep CNN for Multi-Class Image Classification

Design and implement a deep Convolutional Neural Network (CNN) for a multi-class image classification problem using PyTorch. The network should consist of four convolutional blocks, where each block includes two convolution layers followed by activation and normalization, and a pooling layer at the end. The number of filters should increase progressively across blocks. After the convolutional part, flatten the feature maps and connect them to fully connected layers with dropout for regularization, and produce final outputs corresponding to K classes. Also explain the role of each part of the network in detail.

# Import Required Libraries

In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import transforms
from torchvision.datasets import FashionMNIST
from torch.utils.data import DataLoader
import kagglehub

# Download Dataset

In [37]:
path = kagglehub.dataset_download("zalando-research/fashionmnist")
print("Dataset path:", path)

Dataset path: C:\Users\User\.cache\kagglehub\datasets\zalando-research\fashionmnist\versions\4


# Setup - Device and Random Seed

In [38]:
torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using Device: {device}")

Using Device: cpu


# Data Path

In [ ]:
# FashionMNIST is downloaded to path - no custom paths needed
print("Dataset downloaded to:", path)

In [ ]:
class DeepCNN(nn.Module):
    def __init__(self, num_classes):
        super(DeepCNN, self).__init__()

        # Block 1
        self.block1 = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(32),
            nn.Conv2d(32, 32, 3, padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(32),
            nn.MaxPool2d(2)
        )

        # Block 2
        self.block2 = nn.Sequential(
            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(64),
            nn.Conv2d(64, 64, 3, padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(64),
            nn.MaxPool2d(2)
        )

        # Block 3
        self.block3 = nn.Sequential(
            nn.Conv2d(64, 128, 3, padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(128),
            nn.Conv2d(128, 128, 3, padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(128),
            nn.MaxPool2d(2)
        )

        # Block 4
        self.block4 = nn.Sequential(
            nn.Conv2d(128, 256, 3, padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(256),
            nn.Conv2d(256, 256, 3, padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(256),
            nn.MaxPool2d(2)
        )

        self.flatten = nn.Flatten()

        self.fc = nn.Sequential(
            nn.Linear(256 * 8 * 8, 512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.block4(x)
        x = self.flatten(x)
        x = self.fc(x)
        return x

# Data Transformations

In [41]:
transform = transforms.Compose(
    [
        transforms.Resize((128, 128)),
        transforms.Grayscale(num_output_channels=3),  # Convert to 3 channels for CNN
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.5] * 3, std=[0.5] * 3)
    ]
)

In [43]:
train_dataset_full = FashionMNIST(root=path, train=True, transform=transform, download=False)
test_dataset_full = FashionMNIST(root=path, train=False, transform=transform, download=False)
num_classes = len(train_dataset_full.classes)

print("Number of classes:", num_classes)
print("Full train size:", len(train_dataset_full))
print("Full test size:", len(test_dataset_full))

NameError: name 'FashionMNIST' is not defined

# DataLoader

In [ ]:
pin_memory = True if device.type == 'cuda' else False
train_loader = DataLoader(train_dataset_full, batch_size=32, shuffle=True, pin_memory=pin_memory)
test_loader = DataLoader(test_dataset_full, batch_size=32, shuffle=False, pin_memory=pin_memory)

# Deep CNN Model with 4 Convolutional Blocks

## Network Architecture:
- **Block 1**: 2 Conv layers (32 filters) → ReLU → BatchNorm → MaxPool
- **Block 2**: 2 Conv layers (64 filters) → ReLU → BatchNorm → MaxPool
- **Block 3**: 2 Conv layers (128 filters) → ReLU → BatchNorm → MaxPool
- **Block 4**: 2 Conv layers (256 filters) → ReLU → BatchNorm → MaxPool
- **Fully Connected**: Flatten → FC(256*8*8, 512) → ReLU → Dropout → FC(512, 256) → ReLU → Dropout → FC(256, K)

In [ ]:
class DeepCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        
        # Block 1: 2 Conv layers with 32 filters
        self.block1 = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(32),
            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(32),
            nn.MaxPool2d(2)  # 128x128 → 64x64
        )
        
        # Block 2: 2 Conv layers with 64 filters
        self.block2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(64),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(64),
            nn.MaxPool2d(2)  # 64x64 → 32x32
        )
        
        # Block 3: 2 Conv layers with 128 filters
        self.block3 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(128),
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(128),
            nn.MaxPool2d(2)  # 32x32 → 16x16
        )
        
        # Block 4: 2 Conv layers with 256 filters
        self.block4 = nn.Sequential(
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(256),
            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(256),
            nn.MaxPool2d(2)  # 16x16 → 8x8
        )
        
        # Fully Connected Layers
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256 * 8 * 8, 512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(256, num_classes)
        )
    
    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.block4(x)
        x = self.classifier(x)
        return x

# Initialize Model

In [ ]:
model = DeepCNN(num_classes=num_classes).to(device)
print(model)

DeepCNN(
  (block1): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (3): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU()
    (5): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (block2): Sequential(
    (0): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (3): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU()
    (5): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (block3): Sequential(
    (0): Conv2d(64, 128,

# Training Setup

In [ ]:
learning_rate = 0.001
epochs = 10
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.1)

# Training Loop

In [ ]:
for epoch in range(epochs):
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    
    for batch_features, batch_labels in train_loader:
        batch_features = batch_features.to(device)
        batch_labels = batch_labels.to(device)
        
        # Forward pass
        outputs = model(batch_features)
        loss = criterion(outputs, batch_labels)
        
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        total += batch_labels.size(0)
        correct += (predicted == batch_labels).sum().item()
    
    scheduler.step()
    avg_loss = total_loss / len(train_loader)
    accuracy = 100 * correct / total
    print(f"Epoch [{epoch+1}/{epochs}], Loss: {avg_loss:.4f}, Train Accuracy: {accuracy:.2f}%")

KeyboardInterrupt: 

# Evaluation Function

In [ ]:
def evaluate(model, loader):
    model.eval()
    total, correct = 0, 0
    with torch.no_grad():
        for batch_features, batch_labels in loader:
            batch_features = batch_features.to(device)
            batch_labels = batch_labels.to(device)
            outputs = model(batch_features)
            _, predicted = torch.max(outputs, 1)
            total += batch_labels.size(0)
            correct += (predicted == batch_labels).sum().item()
    return 100 * correct / total

In [ ]:
test_accuracy = evaluate(model, test_loader)
print(f"\nTest Accuracy: {test_accuracy:.2f}%")


Test Accuracy: 100.00%


# Explanation of Network Components

## 1. Convolutional Blocks (4 blocks)

Each block contains:
- **Two Convolution Layers**: Extract spatial features from input images. Each layer learns different patterns (edges, textures, shapes).
- **ReLU Activation**: Introduces non-linearity, allowing the network to learn complex patterns.
- **Batch Normalization**: Normalizes activations to stabilize training, speeds up convergence, and provides regularization.
- **Max Pooling**: Reduces spatial dimensions (2x2 with stride 2), reducing computational cost and controlling overfitting.

### Progressive Filter Increase:
- Block 1: 32 filters (capture basic edges)
- Block 2: 64 filters (capture textures)
- Block 3: 128 filters (capture shapes)
- Block 4: 256 filters (capture high-level features)

## 2. Fully Connected Layers

- **Flatten**: Converts 3D feature maps to 1D vector for FC layers.
- **FC Layer 1 (256×8×8 → 512)**: Combines features from all spatial locations.
- **Dropout (0.5)**: Randomly drops neurons during training to prevent overfitting.
- **FC Layer 2 (512 → 256)**: Further abstract the features.
- **Dropout (0.4)**: Additional regularization.
- **Output Layer (256 → K)**: Produces class scores for K classes.

## 3. Key Design Choices

- **Progressive filter increase**: Allows the network to learn increasingly complex features.
- **BatchNorm after each conv block**: Stabilizes training by reducing internal covariate shift.
- **Dropout in FC layers**: Prevents co-adaptation of neurons and reduces overfitting.
- **MaxPool after each block**: Reduces spatial dimensions, making computation efficient.

# Use Subset for Fast Testing

In [ ]:
train_dataset = Subset(train_dataset_full, list(range(min(500, len(train_dataset_full)))))
test_dataset = Subset(test_dataset_full, list(range(min(100, len(test_dataset_full)))))
print(f"Using {len(train_dataset)} training samples")
print(f"Using {len(test_dataset)} test samples")

NameError: name 'train_test_split' is not defined

# Deep Neural Network Model with 4 Hidden Layers

## Network Architecture:
- **Layer 1**: Linear(11, 256) → ReLU → BatchNorm → Dropout
- **Layer 2**: Linear(256, 512) → ReLU → BatchNorm → Dropout
- **Layer 3**: Linear(512, 256) → ReLU → BatchNorm → Dropout
- **Layer 4**: Linear(256, 128) → ReLU → BatchNorm → Dropout
- **Output**: Linear(128, num_classes)

In [ ]:
class DeepNN(nn.Module):
    def __init__(self, input_size, num_classes):
        super().__init__()
        
        # Layer 1: 11 → 256
        self.layer1 = nn.Sequential(
            nn.Linear(input_size, 256),
            nn.ReLU(),
            nn.BatchNorm1d(256),
            nn.Dropout(0.3)
        )
        
        # Layer 2: 256 → 512
        self.layer2 = nn.Sequential(
            nn.Linear(256, 512),
            nn.ReLU(),
            nn.BatchNorm1d(512),
            nn.Dropout(0.3)
        )
        
        # Layer 3: 512 → 256
        self.layer3 = nn.Sequential(
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.BatchNorm1d(256),
            nn.Dropout(0.3)
        )
        
        # Layer 4: 256 → 128
        self.layer4 = nn.Sequential(
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.BatchNorm1d(128),
            nn.Dropout(0.3)
        )
        
        # Output Layer: 128 → num_classes
        self.output = nn.Linear(128, num_classes)
    
    def forward(self, x):
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.output(x)
        return x

# Initialize Model

In [ ]:
input_size = X_train_tensor.shape[1]
model = DeepNN(input_size=input_size, num_classes=num_classes).to(device)
print(model)

# Training Setup

In [ ]:
learning_rate = 0.001
epochs = 50
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)

# Training Loop

In [ ]:
for epoch in range(epochs):
    model.train()
    total_loss = 0
    for batch_features, batch_labels in train_loader:
        batch_features = batch_features.to(device)
        batch_labels = batch_labels.to(device)
        outputs = model(batch_features)
        loss = criterion(outputs, batch_labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1}/{epochs}, Loss: {avg_loss:.4f}")

# Evaluation Function

In [ ]:
def evaluate(model, loader):
    model.eval()
    total, correct = 0, 0
    with torch.no_grad():
        for batch_features, batch_labels in loader:
            batch_features = batch_features.to(device)
            batch_labels = batch_labels.to(device)
            outputs = model(batch_features)
            _, predicted = torch.max(outputs, 1)
            total += batch_labels.size(0)
            correct += (predicted == batch_labels).sum().item()
    return 100 * correct / total

In [ ]:
test_accuracy = evaluate(model, test_loader)
print(f"\nTest Accuracy: {test_accuracy:.2f}%")

# Explanation of Network Components

## 1. Hidden Layers (4 layers)

Each layer contains:
- **Linear Transformation**: Performs matrix multiplication to transform features to higher dimension.
- **ReLU Activation**: Introduces non-linearity, allowing the network to learn complex patterns.
- **Batch Normalization**: Normalizes activations to stabilize training, speeds up convergence.
- **Dropout**: Randomly drops neurons during training to prevent overfitting.

### Progressive Neuron Increase:
- Layer 1: 256 neurons (capture basic patterns)
- Layer 2: 512 neurons (capture more complex patterns)
- Layer 3: 256 neurons (compress features)
- Layer 4: 128 neurons (further compress)

## 2. Output Layer

- **Linear(128, num_classes)**: Produces class scores for K classes.
- **CrossEntropyLoss**: Combines LogSoftmax and NLLLoss for multi-class classification.

## 3. Key Design Choices

- **Progressive neuron increase/decrease**: Allows the network to learn increasingly complex features and then compress them.
- **BatchNorm after each layer**: Stabilizes training by reducing internal covariate shift.
- **Dropout in each layer**: Prevents co-adaptation of neurons and reduces overfitting.
- **ReLU activation**: Efficient and helps mitigate vanishing gradient problem.

In [ ]:
train_dataset_full = MultiClassClassification(TRAIN_PATH, transform)
test_dataset_full = MultiClassClassification(VAL_PATH, transform)
num_classes = len(train_dataset_full.classes)

print(f"Number of classes: {num_classes}")
print(f"Full training set size: {len(train_dataset_full)}")
print(f"Full test set size: {len(test_dataset_full)}")

# Use Subset for Fast Testing

In [ ]:
train_dataset = Subset(train_dataset_full, list(range(min(500, len(train_dataset_full)))))
test_dataset = Subset(test_dataset_full, list(range(min(100, len(test_dataset_full)))))
print(f"Using {len(train_dataset)} training samples")
print(f"Using {len(test_dataset)} test samples")

# DataLoader

In [ ]:
pin_memory = True if device.type == 'cuda' else False
train_loader = DataLoader(train_dataset_full, batch_size=32, shuffle=True, pin_memory=pin_memory)
test_loader = DataLoader(test_dataset_full, batch_size=32, shuffle=False, pin_memory=pin_memory)

# Deep CNN Model with 4 Convolutional Blocks

## Network Architecture:
- **Block 1**: 2 Conv layers (32 filters) → ReLU → BatchNorm → MaxPool
- **Block 2**: 2 Conv layers (64 filters) → ReLU → BatchNorm → MaxPool
- **Block 3**: 2 Conv layers (128 filters) → ReLU → BatchNorm → MaxPool
- **Block 4**: 2 Conv layers (256 filters) → ReLU → BatchNorm → MaxPool
- **Fully Connected**: Flatten → FC(256*4*4, 512) → ReLU → Dropout → FC(512, num_classes)

In [ ]:
class DeepCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        
        # Block 1: 2 Conv layers with 32 filters
        self.block1 = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(32),
            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(32),
            nn.MaxPool2d(2)  # 128x128 → 64x64
        )
        
        # Block 2: 2 Conv layers with 64 filters
        self.block2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(64),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(64),
            nn.MaxPool2d(2)  # 64x64 → 32x32
        )
        
        # Block 3: 2 Conv layers with 128 filters
        self.block3 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(128),
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(128),
            nn.MaxPool2d(2)  # 32x32 → 16x16
        )
        
        # Block 4: 2 Conv layers with 256 filters
        self.block4 = nn.Sequential(
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(256),
            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(256),
            nn.MaxPool2d(2)  # 16x16 → 8x8
        )
        
        # Fully Connected Layers
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256 * 8 * 8, 512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(256, num_classes)
        )
    
    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.block4(x)
        x = self.classifier(x)
        return x

In [ ]:
model = DeepCNN(num_classes=num_classes).to(device)
print(model)

# Training Setup

In [ ]:
learning_rate = 0.001
epochs = 10
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.1)

# Training Loop

In [ ]:
for epoch in range(epochs):
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    
    for batch_features, batch_labels in train_loader:
        batch_features = batch_features.to(device)
        batch_labels = batch_labels.to(device)
        
        # Forward pass
        outputs = model(batch_features)
        loss = criterion(outputs, batch_labels)
        
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        total += batch_labels.size(0)
        correct += (predicted == batch_labels).sum().item()
    
    scheduler.step()
    avg_loss = total_loss / len(train_loader)
    accuracy = 100 * correct / total
    print(f"Epoch [{epoch+1}/{epochs}], Loss: {avg_loss:.4f}, Train Accuracy: {accuracy:.2f}%")

# Evaluation Function

In [ ]:
def evaluate(model, loader):
    model.eval()
    total, correct = 0, 0
    with torch.no_grad():
        for batch_features, batch_labels in loader:
            batch_features = batch_features.to(device)
            batch_labels = batch_labels.to(device)
            outputs = model(batch_features)
            _, predicted = torch.max(outputs, 1)
            total += batch_labels.size(0)
            correct += (predicted == batch_labels).sum().item()
    return 100 * correct / total

In [ ]:
test_accuracy = evaluate(model, test_loader)
print(f"\nTest Accuracy: {test_accuracy:.2f}%")

# Explanation of Network Components

## 1. Convolutional Blocks (4 blocks)

Each block contains:
- **Two Convolution Layers**: Extract spatial features from input images. Each layer learns different patterns (edges, textures, shapes).
- **ReLU Activation**: Introduces non-linearity, allowing the network to learn complex patterns.
- **Batch Normalization**: Normalizes activations to stabilize training, speeds up convergence, and provides regularization.
- **Max Pooling**: Reduces spatial dimensions (2x2 with stride 2), reducing computational cost and controlling overfitting.

### Progressive Filter Increase:
- Block 1: 32 filters (capture basic edges)
- Block 2: 64 filters (capture textures)
- Block 3: 128 filters (capture shapes)
- Block 4: 256 filters (capture high-level features)

## 2. Fully Connected Layers

- **Flatten**: Converts 3D feature maps to 1D vector for FC layers.
- **FC Layer 1 (256×8×8 → 512)**: Combines features from all spatial locations.
- **Dropout (0.5)**: Randomly drops neurons during training to prevent overfitting.
- **FC Layer 2 (512 → 256)**: Further abstract the features.
- **Dropout (0.4)**: Additional regularization.
- **Output Layer (256 → K)**: Produces class scores for K classes.

## 3. Key Design Choices

- **Progressive filter increase**: Allows the network to learn increasingly complex features.
- **BatchNorm after each conv block**: Stabilizes training by reducing internal covariate shift.
- **Dropout in FC layers**: Prevents co-adaptation of neurons and reduces overfitting.
- **MaxPool after each block**: Reduces spatial dimensions, making computation efficient.